# Matching Training Trajectories

推荐你读 https://arxiv.org/pdf/2203.11932 Dataset Distillation by Matching Training Trajectories 这是 MTT 原文。

我想把一件事放在最开头说，那就是 MTT 可能会让人失望。原因是它捡回了 Gradient Matching 抛弃的 Parameter Matching 思想，并且做出了一个类似对于 Curriculum Paramter Matching 的优化的成果。这意味着它具备我们上一章中提到的 PM 多数缺点。

从法理上，首先参数匹配仅仅是模型输出相似的充分而不必要条件。其次的，如果我们考虑整个 Trajectory，我们无可避免地需要对整个轨迹穿透反向传播，这会带来类似原始数据集蒸馏的计算问题。

但是仍有一个好消息，MTT 解决了长程匹配的跨度太大问题。我们详细说。

# 基本逻辑

MTT 的核心思想是用专家轨迹来对齐合成数据集上的轨迹。首先在真实数据集 $\mathcal D_{\mathrm{real}}$ 上训练多个专家轨迹
$$\tau^*
=
\{\theta_t^*\}_{t=0}^{T}$$
对于多个不同初始化专家轨迹，他们是
$$\{\tau_i^*\}$$
这些专家轨迹可以离线预计算，后续所有蒸馏实验共用。

对于每一次蒸馏步骤，我们随机选出一条轨迹 $\tau^*$ 与随机起始时刻 $t$，那么我们可以从专家轨迹中选取参数 
$$\theta_t^*$$

我们将学生参数初始化为此参数 $\hat{\theta}_t=\theta_t^*$。

学生模型更新仅仅使用合成数据集 $\mathcal D_{\mathrm{syn}}$。对于第 $n$ 步更新，其可以写成
$$\hat{\theta}_{t+n+1}
=
\hat{\theta}_{t+n}
-
\alpha
\nabla_\theta
\ell
\left(
\mathcal A(b_{t+n});
\hat{\theta}_{t+n}
\right)$$
其中 $b_{t+n}\sim\mathcal D_{\mathrm{syn}}$ 是从合成数据集取出的 mini-batch，$\mathcal A$ 是我们上一章提到的数据增强算子，$\alpha$ 是学习率。注意，$\alpha$ 也是需要更新的，我们会将其应用在真正的蒸馏数据集学习之中。

所以经历 $N$ 步之后，学生模型抵达
$$\hat{\theta}_{t+N}$$

对于专家轨迹，其从 $\theta_t^*$ 出发进行 $M$ 次更新达到
$$\theta_{t+M}^*$$
我们希望这样一件事
$$\hat{\theta}_{t+N}
\approx
\theta_{t+M}^*$$
这里 $N\ll M$。这个要求乍一看很奇怪但是其实非常合理，我们希望少量合成训练步骤模拟更多真实训练步骤。

最终损失形式是
$$\mathcal L
=
\frac{
\left\|
\hat{\theta}_{t+N}
-
\theta_{t+M}^*
\right\|_2^2
}{
\left\|
\theta_t^*
-
\theta_{t+M}^*
\right\|_2^2
}$$
归一化是必要的，原因是专家轨迹在不同阶段移动距离完全不同。如果我们不用归一化，早期轨迹段可能因为绝对距离大而主导损失。我们希望衡量学生模型相较专家轨迹更新的相对误差。

请注意一下计算图。学生模型最终参数依赖每一步合成数据更新
$$\mathcal D_{\mathrm{syn}}
\longrightarrow
\hat{\theta}_{t+1}
\longrightarrow
\hat{\theta}_{t+2}
\longrightarrow
\cdots
\longrightarrow
\hat{\theta}_{t+N}
\longrightarrow
\mathcal L$$
这意味着对于合成数据集的更新需要穿透 $N$ 个更新步骤，计算 $\nabla_{\mathcal D_{\mathrm{syn}}}\mathcal L$ 并不容易。实际上学习率 $\alpha$ 的更新也大致如此。

# 蒸馏算法

$$\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Dataset Distillation via Trajectory Matching} \\
\hline
\textbf{Input: } \{\tau_i^*\}\text{: set of expert parameter trajectories trained on } \mathcal{D}_{\text{real}}. \\
\textbf{Input: } M\text{: \# of updates between starting and target expert params.} \\
\textbf{Input: } N\text{: \# of updates to student network per distillation step.} \\
\textbf{Input: } \mathcal{A}\text{: Differentiable augmentation function.} \\
\textbf{Input: } T^+ < T\text{: Maximum start epoch.} \\
\begin{aligned}
1: & \ \text{Initialize distilled data } \mathcal{D}_{\text{syn}} \sim \mathcal{D}_{\text{real}} \\
2: & \ \text{Initialize trainable learning rate } \alpha := \alpha_0 \text{ for apply } \mathcal{D}_{\text{syn}} \\
3: & \ \textbf{for each } \text{distillation step\dots} \textbf{ do} \\
4: & \ \quad \triangleright \text{Sample expert trajectory: } \tau^* \sim \{\tau_i^*\} \text{ with } \tau^* = \{\theta_t^*\}_0^T \\
5: & \ \quad \triangleright \text{Choose random start epoch, } t \le T^+ \\
6: & \ \quad \triangleright \text{Initialize student network with expert params:} \\
7: & \ \quad\quad \hat{\theta}_t := \theta_t^* \\
8: & \ \quad \textbf{for } n = 0 \to N - 1 \textbf{ do} \\
9: & \ \quad\quad \triangleright \text{Sample a mini-batch of distilled images:} \\
10:& \ \quad\quad\quad b_{t+n} \sim \mathcal{D}_{\text{syn}} \\
11:& \ \quad\quad \triangleright \text{Update student network w.r.t. classification loss:} \\
12:& \ \quad\quad\quad \hat{\theta}_{t+n+1} = \hat{\theta}_{t+n} - \alpha \nabla \ell(\mathcal{A}(b_{t+n}); \hat{\theta}_{t+n}) \\
13:& \ \quad \textbf{end for} \\
14:& \ \quad \triangleright \text{Compute loss between ending student and expert params:} \\
15:& \ \quad\quad \mathcal{L} = \|\hat{\theta}_{t+N} - \theta_{t+M}^*\|_2^2 \ / \ \|\theta_t^* - \theta_{t+M}^*\|_2^2 \\
16:& \ \quad \triangleright \text{Update } \mathcal{D}_{\text{syn}} \text{ and } \alpha \text{ with respect to } \mathcal{L} \\
17:& \ \textbf{end for}
\end{aligned} \\
\textbf{Output: } \text{distilled data } \mathcal{D}_{\text{syn}} \text{ and learning rate } \alpha \\
\hline
\end{array}$$